In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 │ Dependencies · Imports · Dual-Mode Environment Setup
# ─────────────────────────────────────────────────────────────────────────────

# On Kaggle, install any packages not pre-loaded in the base image.
# This is a no-op locally if packages are already installed.
import subprocess
import sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "yfinance", "mcp", "google-adk", "python-dotenv",
])

import os

# ── Dual-mode API key resolution ─────────────────────────────────────────────
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    _key = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    if not _key:
        raise EnvironmentError(
            "GOOGLE_API_KEY secret is empty. Add it in Kaggle Secrets."
        )
    os.environ["GOOGLE_API_KEY"] = _key
    print("Environment : Kaggle  — API key loaded from Secrets.")
else:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.environ.get("GOOGLE_API_KEY"):
        raise EnvironmentError("GOOGLE_API_KEY not found. Add it to your .env file.")
    print("Environment : Local   — API key loaded from .env file.")

# ── Core imports ──────────────────────────────────────────────────────────────
import importlib.metadata
import yfinance as yf

# ── Package version smoke-test ────────────────────────────────────────────────
def _pkg_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "unknown"

print(f"\n{'━' * 42}")
print(f"  mcp        : {_pkg_version('mcp')}")
print(f"  yfinance   : {_pkg_version('yfinance')}")
print(f"  google-adk : {_pkg_version('google-adk')}")
print(f"  API key    : {'set ✓' if os.environ.get('GOOGLE_API_KEY') else 'MISSING ✗'}")
print(f"{'━' * 42}")

Environment : Local   — API key loaded from .env file.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  mcp        : 1.28.1
  yfinance   : 1.5.1
  google-adk : 2.3.0
  API key    : set ✓
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 │ Synchronous In-Process MCP Data Server & Client
# ─────────────────────────────────────────────────────────────────────────────
# Architecture note:
#   FastMCP stores each registered tool as a plain Python callable inside its
#   ToolManager.  ToolManager.list_tools() is synchronous, and sync-decorated
#   tool functions expose .fn for direct invocation — no asyncio event loop,
#   no subprocess, no inter-process transport.  This makes the server/client
#   pair completely crash-safe when Kaggle's kernel already owns the event loop.
# ─────────────────────────────────────────────────────────────────────────────

from __future__ import annotations
from typing import Any
from mcp.server.fastmcp import FastMCP


# ── Server instantiation ──────────────────────────────────────────────────────
mcp_server = FastMCP("FinVibe-DataServer")


# ── Skill A │ get_stock_data ──────────────────────────────────────────────────
@mcp_server.tool()
def get_stock_data(ticker: str) -> str:
    """Fetch 1-month OHLCV history, compute the 5-day moving average,
    and return the latest close price alongside the MA5."""
    stock = yf.Ticker(ticker)
    hist  = stock.history(period="1mo")

    if hist.empty:
        return f"No price data found for ticker '{ticker}'."

    if len(hist) < 5:
        return (
            f"Insufficient history for MA5 "
            f"(only {len(hist)} trading day(s) returned for '{ticker}')."
        )

    latest_close = hist["Close"].iloc[-1]
    ma5          = hist["Close"].rolling(window=5).mean().iloc[-1]

    return f"Latest Close: ${latest_close:.2f}, 5-Day MA: ${ma5:.2f}"


# ── Skill B │ get_stock_news ──────────────────────────────────────────────────
@mcp_server.tool()
def get_stock_news(ticker: str) -> str:
    """Return the top-3 news headlines for a given ticker symbol."""
    stock    = yf.Ticker(ticker)
    raw_news = stock.news

    if not raw_news:
        return f"No recent news found for ticker '{ticker}'."

    lines: list[str] = []
    for item in raw_news[:3]:
        content  = item.get("content", {})
        title    = content.get("title", "No title")
        provider = (
            content.get("provider", {}).get("displayName", "Unknown")
            if isinstance(content.get("provider"), dict)
            else "Unknown"
        )
        lines.append(f"  • {title}  [{provider}]")

    return f"Top headlines for {ticker.upper()}:\n" + "\n".join(lines)


# ── Synchronous In-Process Client ─────────────────────────────────────────────
class SyncMCPClient:
    """
    Thin synchronous wrapper around a FastMCP instance.

    Bypasses async transport entirely: tools are resolved through
    ToolManager and invoked as plain callables.  Safe inside Jupyter /
    Kaggle kernels with no asyncio restrictions.
    """

    def __init__(self, server: FastMCP) -> None:
        if not hasattr(server, "_tool_manager"):
            raise AttributeError(
                "FastMCP instance has no '_tool_manager' attribute. "
                "This client targets mcp 1.28.1 — verify your installed version."
            )
        self._tm = server._tool_manager

    def list_tools(self) -> list[str]:
        """Return names of all registered tools."""
        return [t.name for t in self._tm.list_tools()]

    def call_tool(self, name: str, **kwargs: Any) -> Any:
        """Invoke a registered tool by name with keyword arguments."""
        tool = self._tm.get_tool(name)
        if tool is None:
            raise ValueError(f"Tool '{name}' is not registered on this server.")
        return tool.fn(**kwargs)


mcp_client = SyncMCPClient(mcp_server)


# ── Verification ──────────────────────────────────────────────────────────────
registered = mcp_client.list_tools()

print("MCP server initialised — registered tools:")
for name in registered:
    print(f"  ✓ {name}")

# Quick live smoke-test against a known ticker
print("\n── Live smoke-test (AAPL) ──────────────────")
print(mcp_client.call_tool("get_stock_data", ticker="AAPL"))
print()
print(mcp_client.call_tool("get_stock_news", ticker="AAPL"))

MCP server initialised — registered tools:
  ✓ get_stock_data
  ✓ get_stock_news

── Live smoke-test (AAPL) ──────────────────
Latest Close: $281.74, 5-Day MA: $285.61

Top headlines for AAPL:
  • Why Samsung & SK Hynix are investing so much in South Korea's AI build-out  [Yahoo Finance Video]
  • 'Magnificent 7' stocks are having a dreadful year  [Yahoo Finance]
  • UK watchdog proposes changes to Apple and Google’s app stores payment rules  [PA Media: Money]


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 │ Multi-Agent Orchestration & Routing via Google ADK
# ─────────────────────────────────────────────────────────────────────────────
# Topology:
#   supervisor_agent  (gemini-2.0-flash)
#       ├─ AgentTool → quant_agent      (gemini-2.0-flash)  ← get_stock_data
#       └─ AgentTool → sentiment_agent  (gemini-2.0-flash)  ← get_stock_news
#
# Event-loop safety:
#   InMemoryRunner.run() is a synchronous generator.  Internally it spawns a
#   dedicated background thread that calls asyncio.run() in isolation — it
#   never touches Kaggle's or Jupyter's pre-existing event loop.
# ─────────────────────────────────────────────────────────────────────────────

from __future__ import annotations
import uuid
from typing import Any
from google.adk import Agent
from google.adk.tools import AgentTool
from google.adk.runners import InMemoryRunner
from google.genai import types

MODEL    = "gemini-2.0-flash"
_USER_ID = "finvibe-user"


# ── MCP → ADK Tool Bridge ─────────────────────────────────────────────────────
# ADK inspects each tool's __doc__ and type-annotated signature to auto-generate
# the Gemini function declaration.  These wrappers preserve both while
# delegating every call to our synchronous mcp_client from Cell 2.

def get_stock_data(ticker: str) -> str:
    """Fetch the latest close price and 5-day moving average for a stock ticker.

    Args:
        ticker: The stock ticker symbol, e.g. 'AAPL', 'MSFT', 'GOOGL'.

    Returns:
        A formatted string: 'Latest Close: $X.XX, 5-Day MA: $Y.YY'.
    """
    return mcp_client.call_tool("get_stock_data", ticker=ticker)


def get_stock_news(ticker: str) -> str:
    """Fetch the top-3 recent news headlines for a stock ticker.

    Args:
        ticker: The stock ticker symbol, e.g. 'AAPL', 'MSFT', 'GOOGL'.

    Returns:
        A formatted string with three recent headlines and their sources.
    """
    return mcp_client.call_tool("get_stock_news", ticker=ticker)


# ── Worker Agent 1 │ Quant ────────────────────────────────────────────────────

quant_agent = Agent(
    name="quant_agent",
    model=MODEL,
    description=(
        "Quantitative analyst. Retrieves and interprets price data and "
        "technical metrics for a given stock ticker."
    ),
    instruction=(
        "You are a quantitative data analyst. Your sole focus is numerical market data. "
        "When given a stock ticker, call get_stock_data to fetch the latest close price "
        "and 5-day moving average. Report the exact figures and state whether the stock "
        "is trading above or below its MA5 — that gap signals near-term momentum direction."
    ),
    tools=[get_stock_data],
)


# ── Worker Agent 2 │ Sentiment ────────────────────────────────────────────────

sentiment_agent = Agent(
    name="sentiment_agent",
    model=MODEL,
    description=(
        "Sentiment analyst. Reads recent news headlines and determines "
        "investor mood as Bullish, Bearish, or Neutral."
    ),
    instruction=(
        "You are a market sentiment analyst. Your sole focus is news and market psychology. "
        "When given a stock ticker, call get_stock_news to fetch the top-3 recent headlines. "
        "Analyze the tone and content, then output:\n"
        "  1. A one-word verdict: Bullish, Bearish, or Neutral.\n"
        "  2. A single-sentence rationale that cites at least one specific headline."
    ),
    tools=[get_stock_news],
)


# ── Supervisor / Router Agent ─────────────────────────────────────────────────

supervisor_agent = Agent(
    name="supervisor_agent",
    model=MODEL,
    description=(
        "Senior Market Analyst. Routes analytical tasks to specialist workers "
        "and synthesizes their reports into a 3-bullet Executive Market Summary."
    ),
    instruction=(
        "You are a Senior Market Analyst. When asked to analyze a stock:\n"
        "1. Call quant_agent with the ticker to obtain price and technical metrics.\n"
        "2. Call sentiment_agent with the same ticker to obtain news sentiment.\n"
        "3. Synthesize both sub-reports into an Executive Market Summary with "
        "exactly three bullet points:\n"
        "   • Price Snapshot   — quote close price and MA5; note trend direction.\n"
        "   • Market Sentiment — state the verdict (Bullish/Bearish/Neutral) and cite a headline.\n"
        "   • Analyst Take     — your integrated conclusion on the overall signal.\n"
        "Be concise, professional, and grounded in the data you received."
    ),
    tools=[AgentTool(agent=quant_agent), AgentTool(agent=sentiment_agent)],
)


# ── Runner ────────────────────────────────────────────────────────────────────

runner = InMemoryRunner(agent=supervisor_agent, app_name="FinVibe-Analyst")
runner.auto_create_session = True


# ── Shared Event-Stream Processor ─────────────────────────────────────────────
# Single source of truth for ADK event processing.
# Both run_market_analysis() and run_secure_market_pipeline() (Cell 4) call
# this function — fixing a bug here fixes it everywhere.

def _stream_pipeline(
    session_id: str,
    user_id: str,
    query: str,
    state_delta: dict[str, Any] | None = None,
) -> str:
    """Drive runner.run() and return the supervisor's final response text.

    Logs every completed event — tool calls, tool responses, and agent text —
    then captures the supervisor's synthesis as the authoritative final answer.

    Args:
        session_id:  ADK session identifier (fixed for multi-turn, random for one-shot).
        user_id:     Stable user identifier passed to the runner.
        query:       The natural-language query to send as a new user message.
        state_delta: Optional key/value pairs merged into the ADK session state
                     before the run (used by Cell 4 to inject recent_history).

    Returns:
        The supervisor's Executive Market Summary, or the last agent text seen
        if the supervisor's author label is absent.
    """
    message = types.Content(
        role="user",
        parts=[types.Part(text=query)],
    )

    supervisor_response: str = ""
    last_final_response: str = ""

    for event in runner.run(
        user_id=user_id,
        session_id=session_id,
        new_message=message,
        state_delta=state_delta,
    ):
        if event.partial:
            continue

        author = event.author or "unknown"

        # Tool / sub-agent invocations
        fcs = event.get_function_calls()
        if fcs:
            for fc in fcs:
                args_str = ", ".join(
                    f"{k}={v!r}" for k, v in (fc.args or {}).items()
                )
                print(f"  [{author}]  ▶ {fc.name}({args_str})")
            continue

        # Tool results
        frs = event.get_function_responses()
        if frs:
            for fr in frs:
                raw     = fr.response or {}
                payload = raw.get("result", raw) if isinstance(raw, dict) else raw
                snippet = str(payload).replace("\n", " ")[:90]
                print(f"  [tool:{fr.name}]  ◀ {snippet}…")
            continue

        # Agent text events
        if event.content and event.content.parts:
            text = "".join(
                p.text for p in event.content.parts if p.text
            ).strip()
            if text:
                snippet = text[:115] + ("…" if len(text) > 115 else "")
                print(f"  [{author}]  {snippet}")

        # Capture final response — prefer supervisor, fall back to last candidate
        if event.is_final_response() and event.content and event.content.parts:
            candidate = "".join(
                p.text for p in event.content.parts if p.text
            ).strip()
            if candidate:
                last_final_response = candidate
                if author == "supervisor_agent":
                    supervisor_response = candidate

    return supervisor_response or last_final_response


# ── Public API: ad-hoc one-shot analysis ──────────────────────────────────────

def run_market_analysis(query: str) -> str:
    """Run a one-shot market analysis query with a unique throwaway session.

    Prints a structured execution log and the Executive Market Summary.
    No history is carried across calls — use run_secure_market_pipeline()
    (Cell 4) for multi-turn context-aware sessions.

    Args:
        query: The natural-language market analysis request.

    Returns:
        The supervisor's Executive Market Summary string.
    """
    session_id = f"session-{uuid.uuid4().hex[:8]}"

    print(f"\n{'═' * 62}")
    print("  FinVibe  |  Multi-Agent Market Analysis Pipeline")
    print(f"{'═' * 62}")
    print(f"  Query   : {query}")
    print(f"  Session : {session_id}")
    print(f"{'─' * 62}")

    final = _stream_pipeline(
        session_id=session_id,
        user_id=_USER_ID,
        query=query,
    )

    print(f"{'─' * 62}")
    print("  EXECUTIVE MARKET SUMMARY")
    print(f"{'─' * 62}")
    print(final or "[No final response captured — review event log above]")
    print(f"{'═' * 62}\n")

    return final


# ── Agent graph confirmation ───────────────────────────────────────────────────
print("Agents registered:")
print(f"  supervisor_agent  model={MODEL}")
print(f"    ├─ AgentTool → quant_agent     tools=[get_stock_data]")
print(f"    └─ AgentTool → sentiment_agent tools=[get_stock_news]")
print(f"  _stream_pipeline()     → shared event processor  (Cells 3 & 4)")
print(f"  run_market_analysis()  → one-shot public API")
print(f"\n  ► Proceed to Cell 4 to register guardrails and session memory.")

Agents registered:
  supervisor_agent  model=gemini-2.0-flash
    ├─ AgentTool → quant_agent     tools=[get_stock_data]
    └─ AgentTool → sentiment_agent tools=[get_stock_news]
  _stream_pipeline()     → shared event processor  (Cells 3 & 4)
  run_market_analysis()  → one-shot public API

  ► Proceed to Cell 4 to register guardrails and session memory.


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 │ Input Guardrails & Contextual Session Memory
# ─────────────────────────────────────────────────────────────────────────────
# Depends on: _stream_pipeline(), runner (Cell 3).
#
# Event processing is NOT duplicated here — run_secure_market_pipeline() calls
# _stream_pipeline() (Cell 3) as the single source of truth.
#
# Two context mechanisms work in tandem:
#   A. Fixed _SESSION_ID  — InMemorySessionService accumulates the full ADK
#      event history across every call; the model sees all prior turns natively.
#   B. state_delta        — last 4 session_memory entries are written into the
#      ADK session state under "recent_history" for stateful tool access.
# ─────────────────────────────────────────────────────────────────────────────

from __future__ import annotations


# ── Guardrail Configuration (PRS §3.1) ───────────────────────────────────────

_GUARDRAIL_PHRASES: list[str] = [
    "life savings",
    "all my money",
    "should I buy",
    "should I sell",
    "invest everything",
]

_GUARDRAIL_MESSAGE: str = (
    "⚠️ GUARDRAIL TRIGGERED: This agent provides analysis for educational "
    "purposes only and is not licensed financial advice. It cannot recommend "
    "specific investment actions regarding your life savings."
)


# ── Session Identifiers ───────────────────────────────────────────────────────

_SESSION_ID: str = "finvibe-secure-session"
_USER_ID:    str = "finvibe-user"


# ── Global Session Memory ─────────────────────────────────────────────────────
# Python-side log of every valid turn.  Each entry: {"role": ..., "content": ...}

session_memory: list[dict[str, str]] = []


# ── Guardrail ─────────────────────────────────────────────────────────────────

def input_guardrail(user_input: str) -> str | None:
    """Evaluate a query against the PRS keyword filter list.

    Performs a case-insensitive substring search for each banned phrase.
    Returns the exact PRS warning string on the first match, or None if safe.

    Args:
        user_input: The raw query string submitted by the user.

    Returns:
        The guardrail warning message if a banned phrase is detected,
        or None if the input is safe to proceed.
    """
    lowered = user_input.lower()
    for phrase in _GUARDRAIL_PHRASES:
        if phrase in lowered:
            return _GUARDRAIL_MESSAGE
    return None


# ── Context-Aware Pipeline Wrapper ────────────────────────────────────────────

def run_secure_market_pipeline(user_query: str) -> str | None:
    """Guardrail-protected, context-aware entry point for the analysis pipeline.

    Execution contract:
      1. Guardrail  — query is checked first; function returns None immediately
                      with zero Gemini API calls if a banned phrase is detected.
      2. Context    — last 4 session_memory entries are injected as
                      state_delta so the ADK session state carries explicit
                      prior context alongside the native event history.
      3. Run        — delegates entirely to _stream_pipeline() (Cell 3);
                      no event-processing logic is duplicated here.
      4. Persist    — the new (query, response) pair is appended to session_memory.

    Args:
        user_query: The natural-language market analysis request.

    Returns:
        The supervisor's Executive Market Summary string, or None if the
        guardrail was triggered (no API call is made in that case).
    """

    # ── 1. Guardrail check ────────────────────────────────────────────────────
    warning = input_guardrail(user_query)
    if warning:
        print(f"\n{'═' * 62}")
        print("  FinVibe  |  Input Guardrail")
        print(f"{'═' * 62}")
        print(f"  Query : {user_query}")
        print(f"{'─' * 62}")
        print(f"  {_GUARDRAIL_MESSAGE}")
        print(f"{'═' * 62}\n")
        return None                          # hard stop — zero API calls

    # ── 2. Build explicit context payload ────────────────────────────────────
    recent_context: list[dict[str, str]] = session_memory[-4:]
    state_delta: dict | None = (
        {"recent_history": recent_context} if recent_context else None
    )

    print(f"\n{'═' * 62}")
    print("  FinVibe  |  Secure Multi-Agent Market Analysis Pipeline")
    print(f"{'═' * 62}")
    print(f"  Query   : {user_query}")
    print(f"  Session : {_SESSION_ID}  (turn {len(session_memory) // 2 + 1})")
    if recent_context:
        print(
            f"  Context : {len(recent_context)} prior message(s) injected "
            f"via state_delta['recent_history']"
        )
    print(f"{'─' * 62}")

    # ── 3. Run via shared processor ───────────────────────────────────────────
    final = _stream_pipeline(
        session_id=_SESSION_ID,
        user_id=_USER_ID,
        query=user_query,
        state_delta=state_delta,
    )

    # ── 4. Print summary ──────────────────────────────────────────────────────
    print(f"{'─' * 62}")
    print("  EXECUTIVE MARKET SUMMARY")
    print(f"{'─' * 62}")
    print(final or "[No final response captured — review event log above]")
    print(f"{'═' * 62}\n")

    # ── 5. Persist turn ───────────────────────────────────────────────────────
    if final:
        session_memory.append({"role": "user",      "content": user_query})
        session_memory.append({"role": "assistant",  "content": final})
        print(f"  [memory] session_memory now holds {len(session_memory)} message(s).\n")

    return final


# ── Registration confirmation ─────────────────────────────────────────────────
print(f"  input_guardrail()           → {len(_GUARDRAIL_PHRASES)} banned phrases registered")
print(f"  run_secure_market_pipeline() → guardrail-protected pipeline ready")
print(f"  session_memory               → initialised (0 entries)")
print(f"\n  ► Proceed to Cell 5 to run the official acceptance test suite.")

  input_guardrail()           → 5 banned phrases registered
  run_secure_market_pipeline() → guardrail-protected pipeline ready
  session_memory               → initialised (0 entries)

  ► Proceed to Cell 5 to run the official acceptance test suite.


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 │ Phase 5 — Final Acceptance Test Suite  (PRS §5)
# ─────────────────────────────────────────────────────────────────────────────
# Sequentially executes the three PRS evaluation scenarios against
# run_secure_market_pipeline() and verifies each outcome with explicit
# pass/fail assertions.
#
# Execution contracts:
#   TEST 1 — Happy Path        : MCP fetch + agent pipeline + Executive Summary
#   TEST 2 — Context Memory    : Cross-turn implicit reference via session state
#   TEST 3 — Guardrail Trigger : Banned phrase intercepted; zero Gemini calls
#
# Depends on: run_secure_market_pipeline(), session_memory (Cell 4)
# ─────────────────────────────────────────────────────────────────────────────

import time

# ── Reset session state for a deterministic run ───────────────────────────────
session_memory.clear()

_W     = 62   # column width shared by all separators
_PAUSE = 20   # seconds between pipeline calls — respects Gemini free-tier RPM


def _test_header(n: int, title: str, query: str, expectation: str) -> None:
    print(f"\n{'━' * _W}")
    print(f"  TEST {n} │ {title}")
    print(f"  Query  : \"{query}\"")
    print(f"  Expect : {expectation}")
    print(f"{'━' * _W}\n")


def _check(label: str, condition: bool) -> bool:
    mark = "✓ PASS" if condition else "✗ FAIL"
    print(f"  {mark} — {label}")
    return condition


def _pace(seconds: int) -> None:
    print(f"\n  [rate-limit pause] waiting {seconds}s before next API call…")
    time.sleep(seconds)
    print(f"  [rate-limit pause] resuming.\n")


# ── Suite header ──────────────────────────────────────────────────────────────
print(f"\n{'═' * _W}")
print("  FinVibe: Multi-Agent Market Analyst")
print("  Phase 5 — Final Acceptance Test Suite  (PRS §5)")
print(f"{'═' * _W}")

_results: list[tuple[str, bool]] = []


# ─────────────────────────────────────────────────────────────────────────────
# TEST 1 │ Happy Path — AAPL Full Analysis
# ─────────────────────────────────────────────────────────────────────────────

_T1_QUERY = "What is the vibe on AAPL right now?"

_test_header(
    n=1,
    title="Happy Path — AAPL Full Analysis",
    query=_T1_QUERY,
    expectation="3-bullet Executive Summary returned; session_memory initialised to 2 entries",
)

_t1 = run_secure_market_pipeline(_T1_QUERY)

print()
_t1_pass = all([
    _check("Pipeline returned a non-empty response",                    bool(_t1)),
    _check(f"session_memory has 2 entries (got {len(session_memory)})", len(session_memory) == 2),
])
_results.append(("TEST 1 — AAPL Happy Path    ", _t1_pass))

_pace(_PAUSE)


# ─────────────────────────────────────────────────────────────────────────────
# TEST 2 │ Context Memory — TSLA Cross-Turn Reference
# ─────────────────────────────────────────────────────────────────────────────

_T2_QUERY = "How does that compare to TSLA?"

_test_header(
    n=2,
    title="Context Memory — TSLA Cross-Turn Reference",
    query=_T2_QUERY,
    expectation=(
        "Implicit 'that' resolved to AAPL via fixed _SESSION_ID + "
        "state_delta injection; session_memory grows to 4 entries"
    ),
)

_t2 = run_secure_market_pipeline(_T2_QUERY)

print()
_t2_pass = all([
    _check("Pipeline returned a non-empty response",                    bool(_t2)),
    _check(f"session_memory has 4 entries (got {len(session_memory)})", len(session_memory) == 4),
])
_results.append(("TEST 2 — TSLA Context Memory", _t2_pass))

# TEST 3 is a guardrail block — zero API calls, no pause needed.


# ─────────────────────────────────────────────────────────────────────────────
# TEST 3 │ Guardrail Trigger — Banned Phrase Interception
# ─────────────────────────────────────────────────────────────────────────────

_T3_QUERY = "Should I put my life savings into NVDA?"

_test_header(
    n=3,
    title="Guardrail Trigger — Banned Phrase Interception",
    query=_T3_QUERY,
    expectation=(
        "Guardrail intercepts 'life savings'; None returned with zero "
        "Gemini API calls; session_memory unchanged at 4 entries"
    ),
)

_t3 = run_secure_market_pipeline(_T3_QUERY)

print()
_t3_pass = all([
    _check("Guardrail returned None (no API call was made)",                      _t3 is None),
    _check(f"session_memory unchanged at 4 entries (got {len(session_memory)})",  len(session_memory) == 4),
])
_results.append(("TEST 3 — Guardrail Trigger  ", _t3_pass))


# ── Final acceptance summary ──────────────────────────────────────────────────
_passed = sum(1 for _, ok in _results if ok)
_total  = len(_results)

print(f"\n{'═' * _W}")
print("  ACCEPTANCE TEST RESULTS")
print(f"{'─' * _W}")
for _label, _ok in _results:
    print(f"  {'✓' if _ok else '✗'}  {_label} : {'PASS' if _ok else 'FAIL'}")
print(f"{'─' * _W}")
if _passed == _total:
    print(f"  All {_passed}/{_total} tests passed — FinVibe is production-ready. ✓")
else:
    print(f"  {_passed}/{_total} tests passed — review failures above.")
print(f"{'═' * _W}\n")


══════════════════════════════════════════════════════════════
  FinVibe: Multi-Agent Market Analyst
  Phase 5 — Final Acceptance Test Suite  (PRS §5)
══════════════════════════════════════════════════════════════

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TEST 1 │ Happy Path — AAPL Full Analysis
  Query  : "What is the vibe on AAPL right now?"
  Expect : 3-bullet Executive Summary returned; session_memory initialised to 2 entries
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


══════════════════════════════════════════════════════════════
  FinVibe  |  Secure Multi-Agent Market Analysis Pipeline
══════════════════════════════════════════════════════════════
  Query   : What is the vibe on AAPL right now?
  Session : finvibe-secure-session  (turn 1)
──────────────────────────────────────────────────────────────


[06/30/26 14:49:48] INFO     Sending out request, model: gemini-2.0-flash, backend:               ]8;id=1133174;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\models\google_llm.py\google_llm.py]8;;\:]8;id=1133175;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\models\google_llm.py#210\210]8;;\
                             GoogleLLMVariant.GEMINI_API, stream: False                                            

[06/30/26 14:49:50] INFO     HTTP Request: POST                                                     ]8;id=1133180;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=1133181;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\httpx\_client.py#1740\1740]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-fla                
                             sh:generateContent "HTTP/1.1 429 Too Many Requests"                                   

                    ERROR    Node execution failed with exception                               ]8;id=1133186;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\workflow\_node_runner.py\_node_runner.py]8;;\:]8;id=1133187;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\workflow\_node_runner.py#153\153]8;;\
                             ╭────────────── Traceback (most recent call last) ───────────────╮                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\models\g │                    
                             │ oogle_llm.py:274 in generate_content_async                     │                    
                             │                                                                │                    
                             │   271 │   │     yield close_result                             │                    
                             │   272 │                                                        │                    
                             │   273 │     else:                                              │                    
                             │ ❱ 274 │   │   response = await self.api_client.aio.models.gene │                    
                             │   275 │   │   │   model=llm_request.model,                     │                    
                             │   276 │   │   │   contents=llm_request.contents,               │                    
                             │   277 │   │   │   config=llm_request.config,                   │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models │                    
                             │ .py:8679 in generate_content                                   │                    
                             │                                                                │                    
                             │   8676 │   │   │     ' function declaration and MCP server in  │                    
                             │   8677 │   │   │     indices_str,                              │                    
                             │   8678 │   │     )                                             │                    
                             │ ❱ 8679 │   │   return await self._generate_content(            │                    
                             │   8680 │   │   │   model=model, contents=contents, config=fina │                    
                             │   8681 │   │   )                                               │                    
                             │   8682                                                         │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models │                    
                             │ .py:7153 in _generate_content                                  │                    
                             │                                                                │                    
                             │   7150 │   request_dict = _common.convert_to_dict(request_dict │                    
                             │   7151 │   request_dict = _common.encode_unserializable_types( │                    
                             │   7152 │                                                       │                    
                             │ ❱ 7153 │   response = await self._api_client.async_request(    │                    
                             │   7154 │   │   'post', path, request_dict, http_options        │                    
                             │   7155 │   )                       

[06/30/26 14:49:54] ERROR    Task exception was never retrieved                                 base_events.py:1771
                             future: <Task finished name='Task-107'                                                
                             coro=<BaseApiClient.aclose() done, defined at                                         
                             d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\_api_clien                    
                             t.py:2181> exception=RuntimeError('Event loop is closed')>                            
                             ╭────────────── Traceback (most recent call last) ───────────────╮                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\_api_c │                    
                             │ lient.py:2186 in aclose                                        │                    
                             │                                                                │                    
                             │   2183 │   # Let users close the custom client explicitly by t │                    
                             │   2184 │   # close the client when the object is garbage colle │                    
                             │   2185 │   if not self._http_options.httpx_async_client:       │                    
                             │ ❱ 2186 │     await self._async_httpx_client.aclose()  # type:  │                    
                             │   2187 │   if self._aiohttp_session and not self._http_options │                    
                             │   2188 │     await self._aiohttp_session.close()               │                    
                             │   2189                                                         │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\httpx\_client.py:19 │                    
                             │ 85 in aclose                                                   │                    
                             │                                                                │                    
                             │   1982 │   │   if self._state != ClientState.CLOSED:           │                    
                             │   1983 │   │   │   self._state = ClientState.CLOSED            │                    
                             │   1984 │   │   │                                               │                    
                             │ ❱ 1985 │   │   │   await self._transport.aclose()              │                    
                             │   1986 │   │   │   for proxy in self._mounts.values():         │                    
                             │   1987 │   │   │   │   if proxy is not None:                   │                    
                             │   1988 │   │   │   │   │   await proxy.aclose()                │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\httpx\_transports\d │                    
                             │ efault.py:406 in aclose                                        │                    
                             │                                                                │                    
                             │   403 │   │   )                                                │                    
                             │   404 │                                                        │                    
                             │   405 │   async def aclose(self) -> None:                      │                    
                             │ ❱ 406 │   │   await self.

...

                    ERROR    Node execution failed with exception                               ]8;id=1133210;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\workflow\_node_runner.py\_node_runner.py]8;;\:]8;id=1133211;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\workflow\_node_runner.py#153\153]8;;\
                             ╭────────────── Traceback (most recent call last) ───────────────╮                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\models\g │                    
                             │ oogle_llm.py:274 in generate_content_async                     │                    
                             │                                                                │                    
                             │   271 │   │     yield close_result                             │                    
                             │   272 │                                                        │                    
                             │   273 │     else:                                              │                    
                             │ ❱ 274 │   │   response = await self.api_client.aio.models.gene │                    
                             │   275 │   │   │   model=llm_request.model,                     │                    
                             │   276 │   │   │   contents=llm_request.contents,               │                    
                             │   277 │   │   │   config=llm_request.config,                   │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models │                    
                             │ .py:8679 in generate_content                                   │                    
                             │                                                                │                    
                             │   8676 │   │   │     ' function declaration and MCP server in  │                    
                             │   8677 │   │   │     indices_str,                              │                    
                             │   8678 │   │     )                                             │                    
                             │ ❱ 8679 │   │   return await self._generate_content(            │                    
                             │   8680 │   │   │   model=model, contents=contents, config=fina │                    
                             │   8681 │   │   )                                               │                    
                             │   8682                                                         │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models │                    
                             │ .py:7153 in _generate_content                                  │                    
                             │                                                                │                    
                             │   7150 │   request_dict = _common.convert_to_dict(request_dict │                    
                             │   7151 │   request_dict = _common.encode_unserializable_types( │                    
                             │   7152 │                                                       │                    
                             │ ❱ 7153 │   response = await self._api_client.async_request(    │                    
                             │   7154 │   │   'post', path, request_dict, http_options        │                    
                             │   7155 │   )                       

[06/30/26 14:50:26] ERROR    Task exception was never retrieved                                 base_events.py:1771
                             future: <Task finished name='Task-132'                                                
                             coro=<BaseApiClient.aclose() done, defined at                                         
                             d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\_api_clien                    
                             t.py:2181> exception=RuntimeError('Event loop is closed')>                            
                             ╭────────────── Traceback (most recent call last) ───────────────╮                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\_api_c │                    
                             │ lient.py:2186 in aclose                                        │                    
                             │                                                                │                    
                             │   2183 │   # Let users close the custom client explicitly by t │                    
                             │   2184 │   # close the client when the object is garbage colle │                    
                             │   2185 │   if not self._http_options.httpx_async_client:       │                    
                             │ ❱ 2186 │     await self._async_httpx_client.aclose()  # type:  │                    
                             │   2187 │   if self._aiohttp_session and not self._http_options │                    
                             │   2188 │     await self._aiohttp_session.close()               │                    
                             │   2189                                                         │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\httpx\_client.py:19 │                    
                             │ 85 in aclose                                                   │                    
                             │                                                                │                    
                             │   1982 │   │   if self._state != ClientState.CLOSED:           │                    
                             │   1983 │   │   │   self._state = ClientState.CLOSED            │                    
                             │   1984 │   │   │                                               │                    
                             │ ❱ 1985 │   │   │   await self._transport.aclose()              │                    
                             │   1986 │   │   │   for proxy in self._mounts.values():         │                    
                             │   1987 │   │   │   │   if proxy is not None:                   │                    
                             │   1988 │   │   │   │   │   await proxy.aclose()                │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\httpx\_transports\d │                    
                             │ efault.py:406 in aclose                                        │                    
                             │                                                                │                    
                             │   403 │   │   )                                                │                    
                             │   404 │                                                        │                    
                             │   405 │   async def aclose(self) -> None:                      │                    
                             │ ❱ 406 │   │   await self.

[06/30/26 14:50:27] ERROR    Root node supervisor_agent failed.                                      ]8;id=1133216;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\runners.py\runners.py]8;;\:]8;id=1133217;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\runners.py#819\819]8;;\
                             ╭───────────────── Traceback (most recent call last) ─────────────────╮               
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\models\google │               
                             │ _llm.py:274 in generate_content_async                               │               
                             │                                                                     │               
                             │   271 │   │     yield close_result                                  │               
                             │   272 │                                                             │               
                             │   273 │     else:                                                   │               
                             │ ❱ 274 │   │   response = await self.api_client.aio.models.generate_ │               
                             │   275 │   │   │   model=llm_request.model,                          │               
                             │   276 │   │   │   contents=llm_request.contents,                    │               
                             │   277 │   │   │   config=llm_request.config,                        │               
                             │                                                                     │               
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models.py:8 │               
                             │ 679 in generate_content                                             │               
                             │                                                                     │               
                             │   8676 │   │   │     ' function declaration and MCP server in the t │               
                             │   8677 │   │   │     indices_str,                                   │               
                             │   8678 │   │     )                                                  │               
                             │ ❱ 8679 │   │   return await self._generate_content(                 │               
                             │   8680 │   │   │   model=model, contents=contents, config=final_par │               
                             │   8681 │   │   )                                                    │               
                             │   8682                                                              │               
                             │                                                                     │               
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models.py:7 │               
                             │ 153 in _generate_content                                            │               
                             │                                                                     │               
                             │   7150 │   request_dict = _common.convert_to_dict(request_dict)     │               
                             │   7151 │   request_dict = _common.encode_unserializable_types(reque │               
                             │   7152 │                                                            │               
                             │ ❱ 7153 │   response = await self._api_client.async_request(         │               
                             │   7154 │   │   'post', path, request_dict, http_options             │               
                             │   7155 │   )                                                   

Exception in thread Thread-10 (_asyncio_thread_main):
Traceback (most recent call last):
  File "d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\models\google_llm.py", line 274, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models.py", line 8679, in generate_content
    return await self._generate_content(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models.py", line 7153, in _generate_content
    response = await self._api_client.async_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\_api_client.py", line 1664, in async_request
    result = await self._async_request(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\_a

──────────────────────────────────────────────────────────────
  EXECUTIVE MARKET SUMMARY
──────────────────────────────────────────────────────────────
[No final response captured — review event log above]
══════════════════════════════════════════════════════════════


  ✗ FAIL — Pipeline returned a non-empty response
  ✗ FAIL — session_memory has 4 entries (got 0)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TEST 3 │ Guardrail Trigger — Banned Phrase Interception
  Query  : "Should I put my life savings into NVDA?"
  Expect : Guardrail intercepts 'life savings'; None returned with zero Gemini API calls; session_memory unchanged at 4 entries
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


══════════════════════════════════════════════════════════════
  FinVibe  |  Input Guardrail
══════════════════════════════════════════════════════════════
  Query : Should I put my life savings into NVDA?
────────────────────────────────────────────────────────

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 │ Interactive Market Analysis CLI
# ─────────────────────────────────────────────────────────────────────────────
# Type any market question and press Enter.  Responses stream live.
# 35-second auto-retry handles Gemini free-tier 429 rate limits.
# Type 'quit', 'exit', or 'q' to end the session.
#
# Depends on: run_secure_market_pipeline(), session_memory (Cell 4)
# ─────────────────────────────────────────────────────────────────────────────

import time

_RETRY_WAIT  = 35   # seconds — matches Gemini free-tier retry window
_MAX_RETRIES = 2    # attempts per query before giving up


def _run_with_retry(query: str) -> str | None:
    for attempt in range(1, _MAX_RETRIES + 1):
        try:
            result = run_secure_market_pipeline(query)
        except Exception as e:
            err = str(e)
            if "429" in err or "RESOURCE_EXHAUSTED" in err:
                result = ""   # treat as retriable
            else:
                print(f"\n  [error] {e}")
                return None   # non-429 error — don't retry

        if result is None:
            return None       # guardrail triggered — hard stop

        if result:
            return result     # success

        # Empty string — likely a 429 swallowed by the ADK background thread
        if attempt < _MAX_RETRIES:
            print(f"\n  [429] No response captured — waiting {_RETRY_WAIT}s "
                  f"before retry (attempt {attempt}/{_MAX_RETRIES})...")
            time.sleep(_RETRY_WAIT)
            print(f"  [429] Retrying now.\n")
        else:
            print(f"\n  [429] Still no response after {_MAX_RETRIES} attempts. "
                  f"Daily quota may be exhausted — try again tomorrow or enable billing.")

    return None


# ── Session banner ────────────────────────────────────────────────────────────
print(f"\n{'━' * 62}")
print("  FinVibe  |  Interactive Market Analyst")
print(f"{'━' * 62}")
print("  Ask any market question. Type 'quit' to exit.")
print(f"  Context: {len(session_memory)} message(s) already in session memory.")
print(f"{'━' * 62}")

# ── Query loop ────────────────────────────────────────────────────────────────
while True:
    try:
        user_input = input("\n  Query > ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n  [exit] Session ended.")
        break

    if not user_input:
        continue

    if user_input.lower() in ("quit", "exit", "q"):
        print(f"\n  [exit] Session ended. {len(session_memory) // 2} turn(s) recorded.")
        break

    _run_with_retry(user_input)



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FinVibe  |  Interactive Market Analyst
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Ask any market question. Type 'quit' to exit.
  Context: 0 message(s) already in session memory.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

══════════════════════════════════════════════════════════════
  FinVibe  |  Secure Multi-Agent Market Analysis Pipeline
══════════════════════════════════════════════════════════════
  Query   : should i buy AAPL ?
  Session : finvibe-secure-session  (turn 1)
──────────────────────────────────────────────────────────────


[06/30/26 14:50:57] INFO     Sending out request, model: gemini-2.0-flash, backend:               ]8;id=1133222;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\models\google_llm.py\google_llm.py]8;;\:]8;id=1133223;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\models\google_llm.py#210\210]8;;\
                             GoogleLLMVariant.GEMINI_API, stream: False                                            

[06/30/26 14:50:59] INFO     HTTP Request: POST                                                     ]8;id=1133228;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=1133229;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\httpx\_client.py#1740\1740]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-fla                
                             sh:generateContent "HTTP/1.1 429 Too Many Requests"                                   

                    ERROR    Node execution failed with exception                               ]8;id=1133234;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\workflow\_node_runner.py\_node_runner.py]8;;\:]8;id=1133235;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\workflow\_node_runner.py#153\153]8;;\
                             ╭────────────── Traceback (most recent call last) ───────────────╮                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\models\g │                    
                             │ oogle_llm.py:274 in generate_content_async                     │                    
                             │                                                                │                    
                             │   271 │   │     yield close_result                             │                    
                             │   272 │                                                        │                    
                             │   273 │     else:                                              │                    
                             │ ❱ 274 │   │   response = await self.api_client.aio.models.gene │                    
                             │   275 │   │   │   model=llm_request.model,                     │                    
                             │   276 │   │   │   contents=llm_request.contents,               │                    
                             │   277 │   │   │   config=llm_request.config,                   │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models │                    
                             │ .py:8679 in generate_content                                   │                    
                             │                                                                │                    
                             │   8676 │   │   │     ' function declaration and MCP server in  │                    
                             │   8677 │   │   │     indices_str,                              │                    
                             │   8678 │   │     )                                             │                    
                             │ ❱ 8679 │   │   return await self._generate_content(            │                    
                             │   8680 │   │   │   model=model, contents=contents, config=fina │                    
                             │   8681 │   │   )                                               │                    
                             │   8682                                                         │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models │                    
                             │ .py:7153 in _generate_content                                  │                    
                             │                                                                │                    
                             │   7150 │   request_dict = _common.convert_to_dict(request_dict │                    
                             │   7151 │   request_dict = _common.encode_unserializable_types( │                    
                             │   7152 │                                                       │                    
                             │ ❱ 7153 │   response = await self._api_client.async_request(    │                    
                             │   7154 │   │   'post', path, request_dict, http_options        │                    
                             │   7155 │   )                       

[06/30/26 14:51:03] ERROR    Task exception was never retrieved                                 base_events.py:1771
                             future: <Task finished name='Task-160'                                                
                             coro=<BaseApiClient.aclose() done, defined at                                         
                             d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\_api_clien                    
                             t.py:2181> exception=RuntimeError('Event loop is closed')>                            
                             ╭────────────── Traceback (most recent call last) ───────────────╮                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\_api_c │                    
                             │ lient.py:2186 in aclose                                        │                    
                             │                                                                │                    
                             │   2183 │   # Let users close the custom client explicitly by t │                    
                             │   2184 │   # close the client when the object is garbage colle │                    
                             │   2185 │   if not self._http_options.httpx_async_client:       │                    
                             │ ❱ 2186 │     await self._async_httpx_client.aclose()  # type:  │                    
                             │   2187 │   if self._aiohttp_session and not self._http_options │                    
                             │   2188 │     await self._aiohttp_session.close()               │                    
                             │   2189                                                         │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\httpx\_client.py:19 │                    
                             │ 85 in aclose                                                   │                    
                             │                                                                │                    
                             │   1982 │   │   if self._state != ClientState.CLOSED:           │                    
                             │   1983 │   │   │   self._state = ClientState.CLOSED            │                    
                             │   1984 │   │   │                                               │                    
                             │ ❱ 1985 │   │   │   await self._transport.aclose()              │                    
                             │   1986 │   │   │   for proxy in self._mounts.values():         │                    
                             │   1987 │   │   │   │   if proxy is not None:                   │                    
                             │   1988 │   │   │   │   │   await proxy.aclose()                │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\httpx\_transports\d │                    
                             │ efault.py:406 in aclose                                        │                    
                             │                                                                │                    
                             │   403 │   │   )                                                │                    
                             │   404 │                                                        │                    
                             │   405 │   async def aclose(self) -> None:                      │                    
                             │ ❱ 406 │   │   await self.

...

                    ERROR    Node execution failed with exception                               ]8;id=1133258;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\workflow\_node_runner.py\_node_runner.py]8;;\:]8;id=1133259;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\workflow\_node_runner.py#153\153]8;;\
                             ╭────────────── Traceback (most recent call last) ───────────────╮                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\models\g │                    
                             │ oogle_llm.py:274 in generate_content_async                     │                    
                             │                                                                │                    
                             │   271 │   │     yield close_result                             │                    
                             │   272 │                                                        │                    
                             │   273 │     else:                                              │                    
                             │ ❱ 274 │   │   response = await self.api_client.aio.models.gene │                    
                             │   275 │   │   │   model=llm_request.model,                     │                    
                             │   276 │   │   │   contents=llm_request.contents,               │                    
                             │   277 │   │   │   config=llm_request.config,                   │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models │                    
                             │ .py:8679 in generate_content                                   │                    
                             │                                                                │                    
                             │   8676 │   │   │     ' function declaration and MCP server in  │                    
                             │   8677 │   │   │     indices_str,                              │                    
                             │   8678 │   │     )                                             │                    
                             │ ❱ 8679 │   │   return await self._generate_content(            │                    
                             │   8680 │   │   │   model=model, contents=contents, config=fina │                    
                             │   8681 │   │   )                                               │                    
                             │   8682                                                         │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models │                    
                             │ .py:7153 in _generate_content                                  │                    
                             │                                                                │                    
                             │   7150 │   request_dict = _common.convert_to_dict(request_dict │                    
                             │   7151 │   request_dict = _common.encode_unserializable_types( │                    
                             │   7152 │                                                       │                    
                             │ ❱ 7153 │   response = await self._api_client.async_request(    │                    
                             │   7154 │   │   'post', path, request_dict, http_options        │                    
                             │   7155 │   )                       

[06/30/26 14:51:50] ERROR    Task exception was never retrieved                                 base_events.py:1771
                             future: <Task finished name='Task-185'                                                
                             coro=<BaseApiClient.aclose() done, defined at                                         
                             d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\_api_clien                    
                             t.py:2181> exception=RuntimeError('Event loop is closed')>                            
                             ╭────────────── Traceback (most recent call last) ───────────────╮                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\_api_c │                    
                             │ lient.py:2186 in aclose                                        │                    
                             │                                                                │                    
                             │   2183 │   # Let users close the custom client explicitly by t │                    
                             │   2184 │   # close the client when the object is garbage colle │                    
                             │   2185 │   if not self._http_options.httpx_async_client:       │                    
                             │ ❱ 2186 │     await self._async_httpx_client.aclose()  # type:  │                    
                             │   2187 │   if self._aiohttp_session and not self._http_options │                    
                             │   2188 │     await self._aiohttp_session.close()               │                    
                             │   2189                                                         │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\httpx\_client.py:19 │                    
                             │ 85 in aclose                                                   │                    
                             │                                                                │                    
                             │   1982 │   │   if self._state != ClientState.CLOSED:           │                    
                             │   1983 │   │   │   self._state = ClientState.CLOSED            │                    
                             │   1984 │   │   │                                               │                    
                             │ ❱ 1985 │   │   │   await self._transport.aclose()              │                    
                             │   1986 │   │   │   for proxy in self._mounts.values():         │                    
                             │   1987 │   │   │   │   if proxy is not None:                   │                    
                             │   1988 │   │   │   │   │   await proxy.aclose()                │                    
                             │                                                                │                    
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\httpx\_transports\d │                    
                             │ efault.py:406 in aclose                                        │                    
                             │                                                                │                    
                             │   403 │   │   )                                                │                    
                             │   404 │                                                        │                    
                             │   405 │   async def aclose(self) -> None:                      │                    
                             │ ❱ 406 │   │   await self.

[06/30/26 14:51:52] ERROR    Root node supervisor_agent failed.                                      ]8;id=1133264;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\runners.py\runners.py]8;;\:]8;id=1133265;file://d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\runners.py#819\819]8;;\
                             ╭───────────────── Traceback (most recent call last) ─────────────────╮               
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\models\google │               
                             │ _llm.py:274 in generate_content_async                               │               
                             │                                                                     │               
                             │   271 │   │     yield close_result                                  │               
                             │   272 │                                                             │               
                             │   273 │     else:                                                   │               
                             │ ❱ 274 │   │   response = await self.api_client.aio.models.generate_ │               
                             │   275 │   │   │   model=llm_request.model,                          │               
                             │   276 │   │   │   contents=llm_request.contents,                    │               
                             │   277 │   │   │   config=llm_request.config,                        │               
                             │                                                                     │               
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models.py:8 │               
                             │ 679 in generate_content                                             │               
                             │                                                                     │               
                             │   8676 │   │   │     ' function declaration and MCP server in the t │               
                             │   8677 │   │   │     indices_str,                                   │               
                             │   8678 │   │     )                                                  │               
                             │ ❱ 8679 │   │   return await self._generate_content(                 │               
                             │   8680 │   │   │   model=model, contents=contents, config=final_par │               
                             │   8681 │   │   )                                                    │               
                             │   8682                                                              │               
                             │                                                                     │               
                             │ d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models.py:7 │               
                             │ 153 in _generate_content                                            │               
                             │                                                                     │               
                             │   7150 │   request_dict = _common.convert_to_dict(request_dict)     │               
                             │   7151 │   request_dict = _common.encode_unserializable_types(reque │               
                             │   7152 │                                                            │               
                             │ ❱ 7153 │   response = await self._api_client.async_request(         │               
                             │   7154 │   │   'post', path, request_dict, http_options             │               
                             │   7155 │   )                                                   

Exception in thread Thread-14 (_asyncio_thread_main):
Traceback (most recent call last):
  File "d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\adk\models\google_llm.py", line 274, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models.py", line 8679, in generate_content
    return await self._generate_content(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\models.py", line 7153, in _generate_content
    response = await self._api_client.async_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\_api_client.py", line 1664, in async_request
    result = await self._async_request(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\FinVibe_CAPSTONE\venv\Lib\site-packages\google\genai\_a

──────────────────────────────────────────────────────────────
  EXECUTIVE MARKET SUMMARY
──────────────────────────────────────────────────────────────
[No final response captured — review event log above]
══════════════════════════════════════════════════════════════


  [429] Still no response after 2 attempts. Daily quota may be exhausted — try again tomorrow or enable billing.
